# DS2002 · Midterm Clinic Cleaning and Joins

**Lecture — 2026-10-26 · Fall 2026**  
**Class time:** 45 minutes

---

## Clinic — cleaning and joins

Week 2 of 3. No new material today. Bring your team's actual notebook and use the hour to get unstuck.

Below are the six problems that came up most in the week-1 checkpoints, each with a runnable fix you can adapt. Find yours, take the pattern, and get back to your data.

If none of these is your problem, come find me — that is what the hour is for.

### 1. Fragmented SKUs, with a mapping table instead of a dict

A dictionary works for a dozen products. For a real catalog you want the mapping as a **table**, because a table can be committed, reviewed, and edited by whoever knows the products — and a left join tells you which SKUs you have not mapped yet.

In [ ]:
import pandas as pd, numpy as np

tx = pd.DataFrame({
    'sku': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'WATER-24', 'FLASH-AA'],
    'qty': [40, 15, 22, 60, 8],
})
mapping = pd.DataFrame({
    'raw_sku': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'WATER-24'],
    'canonical': ['POPTART-STRAW', 'POPTART-STRAW', 'POPTART-STRAW', 'WATER-24'],
})

joined = tx.merge(mapping, left_on='sku', right_on='raw_sku', how='left')
joined['canonical'] = joined['canonical'].fillna(joined['sku'])

unmapped = joined.loc[joined['raw_sku'].isna(), 'sku'].tolist()
print('not yet in the mapping table:', unmapped)
print()
print(joined.groupby('canonical')['qty'].sum())
assert joined['qty'].sum() == tx['qty'].sum(), 'mapping changed unit totals'

The `unmapped` list is the useful part. It is your to-do list, generated from the data, and it shrinks as you work instead of you wondering whether you got everything.

### 2. The join that duplicated your rows

If your revenue total jumped after a merge, your right-hand table has duplicate keys. Check before you join, not after you present.

In [ ]:
stores = pd.DataFrame({
    'store_id': ['FL-105', 'FL-239', 'FL-105'],   # duplicated!
    'city': ['Orlando', 'Tampa', 'Orlando'],
})

print('is store_id unique?', stores['store_id'].is_unique)
print('duplicated keys:', stores.loc[stores['store_id'].duplicated(), 'store_id'].tolist())

safe = stores.drop_duplicates(subset='store_id')
assert safe['store_id'].is_unique
print('safe to join on:', safe['store_id'].tolist())

# Or let pandas police it for you -- this raises instead of silently duplicating:
sales = pd.DataFrame({'store_id': ['FL-105', 'FL-239'], 'revenue': [100, 200]})
try:
    sales.merge(stores, on='store_id', validate='many_to_one')
except Exception as e:
    print('\nvalidate caught it:', type(e).__name__, '->', e)

`validate='many_to_one'` is the single most useful argument in `merge` and almost nobody uses it. Add it to every join in your project.

### 3. Baseline vs surge windows

Question 1 needs a baseline. The window boundaries are a decision you have to state, because the multiple you report depends on them.

In [ ]:
daily = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=14),
    'orders': [100, 110, 95, 105, 120, 340, 520, 480,
               90, 100, 110, 105, 98, 102],
})
landfall = pd.Timestamp('2024-09-07')

for window in (2, 3, 5):
    start = landfall - pd.Timedelta(days=window)
    surge = daily[(daily['date'] >= start) & (daily['date'] <= landfall)]
    baseline = daily[daily['date'] < start]
    print(f'{window}-day window: surge {surge["orders"].mean():6.1f} | '
          f'baseline {baseline["orders"].mean():6.1f} | '
          f'{surge["orders"].mean() / baseline["orders"].mean():.1f}x')

Three defensible windows, three different multiples, from 2.6x to 3.6x. None of them is wrong. What would be wrong is reporting one without saying which window you used.

Pick your window for a stated reason — "the storm entered the forecast three days out" is a reason — and put that sentence in the notebook.

### 4. Timestamps that will not parse

If `pd.to_datetime` is failing or producing surprising dates, look at the values it choked on rather than reaching for a bigger hammer.

In [ ]:
messy = pd.Series(['2024-09-06 10:06:00', '09/06/2024 10:40',
                   '2024-09-06T11:00:00', '', 'Sept 6 2024', '06/09/2024'])

parsed = pd.to_datetime(messy, errors='coerce', format='mixed')
failed = messy[parsed.isna()]
print('failed to parse:')
print(failed.tolist())
print()
print(parsed)

Two things to notice. The empty string fails, which is correct — there is no date there. And `06/09/2024` parsed as **June 9th**, not September 6th, because pandas assumed US month-first ordering.

That second one will not raise an error and will not look wrong. If your source mixes day-first and month-first formats, you have to determine which is which from the data — look for values above 12 in the first position — and parse them in separate passes.

### 5. Store ids in four spellings

Normalize before you group or join, or the same store shows up as four stores.

In [ ]:
ids = pd.Series(['FL-105', 'fl-105', 'FL 105', 'fl105', 'FL-239'])

normalized = (ids.str.upper().str.strip()
              .str.replace(r'[^A-Z0-9]', '', regex=True)
              .apply(lambda s: f'{s[:2]}-{s[2:]}'))

for before, after in zip(ids, normalized):
    print(f'{before:10s} -> {after}')
print()
print('distinct stores:', ids.nunique(), '->', normalized.nunique())

### 6. "Our numbers changed and we do not know why"

This is a version-control problem, not a pandas problem. If two people cleaned the same data in different notebooks, you now have two answers and no way to reconcile them.

The fix, today: one cleaning module, one owner, everyone imports the result. Put the cleaning functions in a single notebook or `.py` file, have one person own it this week, and write the row count after cleaning into your decision log. If the row count moves, somebody changed a decision and it needs to be discussed rather than discovered.

### Before you leave

- Every merge in your notebook has a row-count check or `validate=`
- Your baseline window is chosen and the reason is written down
- Your decision log has a row count on every line
- Everything is pushed, and one person is named as this week's notebook owner

**This week's target:** Questions 1 through 3 substantially answered. That is what Friday's progress check asks for.